[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/navjotts/ML-experiments/blob/master/23%20-%20Recommendation%20System%20using%20Matrix%20Factorization/Recommendation_System_using_Matrix_Factorization.ipynb)

# Overview

**1)** **Content-based Recommendations** i.e. recommend products/items based on the attibutes of the product/item

Content-based recommendation engines utilize their knowledge of the attibutes of each product/item to recommend new products/items. 

Let's say that you inform your colleague at work that you just watched the movie **Rock** starring *Nicholas Cage* and that you really liked the movie. Based on this information, your colleague might recommend that you watch the movie **Con Air** next. Both movies are Action-packed Thrillers and both movies feature the same movie star. It could be a good recommendation because the movies have a few attributes in common. This is the fundamental idea behind content-based recommendation engines. Content-based recommendation engines recommend products that have similar attributes to a product that the user already liked.

**Point to Note**: Content-based recommendations can be effective only if you have *descriptive data* available for each product/item that you want to recommend. However, creating detailed information for each product/item in your inventory is not only a time-consuming process but it also introduces a fair degree of subjectivity that can throw off your recommendation results.

**2)** **Collaborative Filtering** i.e. recommend products solely based on user ratings

Collaborative filtering recommendation engines generate recommendations solely based on how users rated products in the past; it only possesses knowledge on how other users rated the product and it uses those past ratings to make new recommendations.

**Advantage** over content-based recommendations: No knowledge about the attributes of the products/items being recommended is required. However if there is no information on user reviews/ratings, recommendations cannot be made

**Limitations:** Collaborative Filtering only works when you already have user reviews to work from. If you do not have any reviews, you cannot make recommendations. Also, collaborative filtering tends to favor items with lots of reviews over items with few reviews. This can make it difficult for users to uncover new releases since they are not likely to get recommended as often.




#  Matrix Factorization

### Utilizing Matrix Factorization to estimate/predict missing user ratings

To develop a framework for making recommendations, we will walk-through an example:

**Goal**: Predict missing user ratings so that recommendations about similar products/items (i.e. movies for the purposes of this example) can be made

**Available Data**: **a)** User ratings for movies, **b)** List of movie titles and the respective genres

**Process**: In order to compute missing user ratings, we need 2 key pieces of information:

1) the user preferences across a range of attributes (the attributes could encompass Action, Comedy, Romance, Horror, etc.) 

2) the ratings for *each* movie across the same set of attributes

We will utilize the "User ratings for movies" data set to factor out a "User Ratings" matrix as well as a "Movie Rating" matrix. Once we have the 2 pieces of information highlighted above, we will leverage the concepts of Linear Algebra (i.e. **Matrix Multiplication**) to arrive at a value for "User Movie Rating" for each movie across every user which will provide an estimate/prediction for the missing user ratings so that the appropriate recommendations can be surfaced to each user

**User Movie Ratings** = User Ratings (across attributes) **X** Movie Ratings (across the same set of attributes) 

*Point to note*: When computing the following matrices: User Ratings (across attributes) and the Movie Ratings (across attributes), we have no idea what each attribute/feature is. All we know is that each attribute/feature represents some characteristic that has attracted users to certain products/items (in this case movies). Since we are unsure of how to describe those characteristics in words, they are deemed as **latent** features. The word latent just means hidden. 



In [0]:
import numpy as np
import pandas as pd
from pandas import Series, DataFrame


df = pd.read_csv('https://www.dropbox.com/s/bshafyt7nne3n1l/movie_ratings.csv?raw=1')
df.head()

,userid,movieid,rating
0,1,28,4
1,1,26,4
2,1,9,4
3,1,1,4
4,1,14,4


In [0]:
# Transpose the dataset set so that the columns represent the individual movies
df_pivot = pd.pivot_table(df, index='userid', columns='movieid', aggfunc=np.max).fillna(0)
df_pivot

rating                                              ...             \
movieid     1    2    3    4    5    6    7    8    9    10 ...    25   26   
userid                                                      ...              
1          4.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  4.0  0.0 ...   0.0  4.0   
2          5.0  5.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0 ...   0.0  0.0   
3          4.0  4.0  5.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0 ...   0.0  0.0   
4          5.0  5.0  0.0  5.0  5.0  0.0  0.0  0.0  0.0  0.0 ...   0.0  0.0   
5          5.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  5.0  0.0 ...   0.0  0.0   
6          5.0  5.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0 ...   0.0  0.0   
7          5.0  0.0  0.0  2.0  0.0  0.0  0.0  0.0  0.0  0.0 ...   0.0  0.0   
8          4.0  0.0  5.0  0.0  0.0  0.0  0.0  0.0  0.0  5.0 ...   0.0  0.0   
9          5.0  0.0  5.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0 ...   0.0  0.0   
10         4.0  0.0  4.0  0.0  0.0  0.0  0.0  4.0  0.0  0.0 ...   0.0  0.0   
11         5.0  4.0  5.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0 ...   0.0  0.0   
12         5.0  0.0  5.0  0.0  0.0  0.0  0.0  5.0  0.0  0.0 ...   0.0  0.0   
13         4.0  0.0  0.0  0.0  0.0  0.0  5.0  0.0  0.0  0.0 ...   0.0  0.0   
14         5.0  4.0  5.0  0.0  5.0  0.0  0.0  0.0  0.0  0.0 ...   0.0  0.0   
15         5.0  0.0  5.0  5.0  0.0  0.0  0.0  0.0  0.0  0.0 ...   0.0  0.0   
16         4.0  0.0  4.0  0.0  0.0  0.0  0.0  4.0  0.0  0.0 ...   0.0  0.0   
17         4.0  4.0  4.0  0.0  5.0  0.0  0.0  0.0  0.0  0.0 ...   0.0  0.0   
18         5.0  0.0  5.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0 ...   5.0  0.0   
19         0.0  5.0  0.0  0.0  0.0  5.0  0.0  0.0  5.0  0.0 ...   5.0  0.0   
20         0.0  5.0  0.0  0.0  5.0  0.0  0.0  0.0  0.0  0.0 ...   0.0  0.0   
21         0.0  4.0  0.0  0.0  3.0  0.0  0.0  3.0  0.0  0.0 ...   0.0  4.0   
22         0.0  4.0  0.0  0.0  0.0  0.0  4.0  0.0  4.0  0.0 ...   0.0  0.0   
23         0.0  4.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0 ...   4.0  0.0   
24         0.0  5.0  0.0  0.0  5.0  0.0  0.0  0.0  0.0  5.0 ...   0.0  0.0   
25         0.0  5.0  4.0  0.0  0.0  0.0  0.0  0.0  0.0  5.0 ...   0.0  0.0   
26         0.0  5.0  0.0  0.0  5.0  5.0  0.0  5.0  0.0  5.0 ...   0.0  0.0   
27         0.0  5.0  0.0  0.0  5.0  0.0  0.0  0.0  0.0  0.0 ...   0.0  5.0   
28         0.0  5.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0 ...   5.0  0.0   
29         0.0  5.0  0.0  0.0  5.0  0.0  0.0  0.0  0.0  0.0 ...   0.0  0.0   
30         0.0  0.0  5.0  0.0  5.0  0.0  0.0  0.0  0.0  5.0 ...   0.0  0.0   
...        ...  ...  ...  ...  ...  ...  ...  ...  ...  ... ...   ...  ...   
71         0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0 ...   5.0  0.0   
72         0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0 ...   0.0  0.0   
73         0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0 ...   0.0  0.0   
74         0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0 ...   0.0  0.0   
75         0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0 ...   0.0  0.0   
76         0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0 ...   0.0  0.0   
77         0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0 ...   0.0  0.0   
78         0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0 ...   0.0  0.0   
79         0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0 ...   0.0  0.0   
80         0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0 ...   0.0  0.0   
81         0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0 ...   0.0  0.0   
82         0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0 ...   0.0  4.0   
83         0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0 ...   0.0  0.0   
84         0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0 ...   0.0  0.0   
85         0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0 ...   0.0  0.0   
86         0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0 ...   0.0  0.0   
87         0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0 ...   0.0  0.0   
88         0.0  0.0  0.0  0.0 

We will utilize ** Matrix Factorization** to arrive at the 2 matrices i.e. **a)** User Ratings (across attributes)** b) ** Movie Ratings (across attributes). Once we have the 2 matrices, we will compute the "dot" product of the 2 matrices to come up with an estimate/prediction for the  missing user ratings 

Reference: http://www.quuxlabs.com/wp-content/uploads/2010/09/mf.py_.txt




In [0]:
#
# Created by Albert Au Yeung (2010)
#
# An implementation of matrix factorization
#

def matrix_factorization(R, P, Q, K, steps=5000, alpha=0.0002, beta=0.02):
    Q = Q.T
    for step in range(steps):
        for i in range(len(R)):
            for j in range(len(R[i])):
                if R[i][j] > 0:
                    eij = R[i][j] - np.dot(P[i,:],Q[:,j])
                    for k in range(K):
                        P[i][k] = P[i][k] + alpha * (2 * eij * Q[k][j] - beta * P[i][k])
                        Q[k][j] = Q[k][j] + alpha * (2 * eij * P[i][k] - beta * Q[k][j])
        eR = np.dot(P,Q)
        e = 0
        for i in range(len(R)):
            for j in range(len(R[i])):
                if R[i][j] > 0:
                    e = e + pow(R[i][j] - np.dot(P[i,:],Q[:,j]), 2)
                    for k in range(K):
                        e = e + (beta/2) * ( pow(P[i][k],2) + pow(Q[k][j],2) )
        if e < 0.001:
            break
    return P, Q.T

In [0]:
# R     : a matrix to be factorized, dimension N x M
# P     : an initial matrix of dimension N x K
# Q     : an initial matrix of dimension M x K
# K     : the number of latent features
# steps : the maximum number of steps to perform the optimization
# alpha : the learning rate
# beta  : the regularization parameter  
R = df_pivot.as_matrix()
N = len(R)
M = len(R[0])
K = 2

P = np.random.rand(N,K) # Matrix of user attributes
Q = np.random.rand(M,K) # Matrix of movie attributes

nP, nQ = matrix_factorization(R, P, Q, K)

In [0]:
# User Movie Ratings = User Ratings (across attributes) **X** Movie Ratings (across the same set of attributes) 

nR = np.dot(nP, nQ.T)

print(nR) # Estimate/Prediction for the user movie ratings
print(nR.shape)
print(nR[0,:]) # Ratings from first user

[[4.18932986 3.88137326 4.31552114 ... 3.67293771 3.34122553 3.61040093]
 [4.72019035 4.79914629 4.95240166 ... 4.26354741 4.137008   4.46967355]
 [4.10304306 4.28257326 4.32833382 ... 3.73869787 3.69306615 3.9898866 ]
 ...
 [5.58443492 5.44146061 5.80919834 ... 4.97470741 4.68780416 5.06507329]
 [4.15651056 3.45166307 4.19731313 ... 3.52680628 2.96594339 3.2054662 ]
 [4.52023017 3.60392988 4.53294564 ... 3.79140344 3.094535   3.34468608]]
(100, 34)
[4.18932986 3.88137326 4.31552114 3.75377845 3.94793587 3.93286625
 3.64977826 3.89917156 4.07182785 4.01638838 3.85529034 3.99477042
 4.21502935 4.05147929 2.9894008  3.93715872 2.78874719 3.04302281
 3.47707751 3.59319636 3.64829024 3.95777419 3.4875902  3.92990157
 3.68197882 4.27202183 3.18789222 4.15230335 3.09185111 2.74784318
 3.21549787 3.67293771 3.34122553 3.61040093]


# Measure Recommendation Accuracy

To measure the accuracy of estimated "User Movie Ratings" , we will use a  statistical metric called root-mean-square-error (RMSE). RMSE is a meaure of the difference between the user's *actual *movie ratings and the ratings that were predicted for the same movies. 

The lower the root-mean-square-error, the more accurate the model. A root-mean-square-error of zero means our model perfectly estimates user ratings. If the root-mean-square-error for the test set is much higher than that of the training set, it is likely that you may have overfitted the data. In this example scenario, we are off by about one rating star on average when predicting user ratings.

We could adjust the **regularization** amount parameter to improve the accuracy of the recommendations. Regularization limits the amount of weight we place on a single attribute when finding user/item features with matrix factorization. The higher we set the regularization amount, the less weight we put on any single attribute. When you are building a recommendation engine, you will want to experiment with different regularization values to see how it affects the quality of your recommendations.

**Note**: One limitation that we faced in this example scenario is that we only had a few hundred movie reviews to work with; the best thing that could be done to improve accuracy in this case is to get more user reviews. More movie reviews will give our engine more information to work with so it can do a better job of making recommendations.





In [0]:
import numpy as np
import pandas as pd
from pandas import Series, DataFrame

df_training = pd.read_csv('https://www.dropbox.com/s/32kkiklfavh97ad/movie_ratings_training.csv?raw=1')
df_testing = pd.read_csv('https://www.dropbox.com/s/pjbv00xjv0w8354/movie_ratings_testing.csv?raw=1')

In [0]:
print(df_training.shape)
userIds = df_training['userid'].unique()
userIds.sort()
movieIds = df_training['movieid'].unique()
movieIds.sort()
print(userIds)
print(movieIds)
df_training.head()

(475, 3)
[  1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17  18
  19  20  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35  36
  37  38  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53  54
  55  56  57  58  59  60  61  62  63  64  65  66  67  68  69  70  71  72
  73  74  75  76  77  78  79  80  81  82  83  84  85  86  87  88  89  90
  91  92  93  94  95  96  97  98  99 100]
[ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24
 25 26 27 28 29 30 31 32 33 34]


,userid,movieid,rating
0,97,30,5
1,31,9,5
2,73,32,3
3,31,3,5
4,73,20,2


In [0]:
print(df_testing.shape)
userIds = df_testing['userid'].unique()
userIds.sort()
movieIds = df_testing['movieid'].unique()
movieIds.sort()
print(userIds)
print(movieIds)
df_testing.head()

(211, 3)
[  1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17  18
  19  20  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35  36
  37  38  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53  54
  55  56  57  58  59  60  61  62  63  64  65  66  67  68  69  70  71  72
  73  74  75  76  77  78  79  80  81  82  83  84  85  86  87  88  89  90
  91  92  93  94  95  96  97  98  99 100]
[ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24
 25 26 27 28 29 30 31 32 33 34]


,userid,movieid,rating
0,32,13,5
1,26,8,5
2,70,30,5
3,66,19,4
4,38,3,5


In [0]:
df_training_pivot = pd.pivot_table(df_training, index='userid', columns='movieid', aggfunc=np.max)
df_testing_pivot = pd.pivot_table(df_testing, index='userid', columns='movieid', aggfunc=np.max)

**Let's try to see if we can optimize the error by adjusting regularization**

In [0]:
def RMSE(real, predicted):
  return np.sqrt(np.nanmean(np.square(real - predicted)))

def calculate_recommendation_error(beta):
  R = df_training_pivot.as_matrix()
  N = len(R)
  M = len(R[0])
  K = 2

  P = np.random.rand(N,K) # Matrix of user attributes
  Q = np.random.rand(M,K) # Matrix of movie attributes

  nP, nQ = matrix_factorization(R, P, Q, K, beta=beta)

  # Find all predicted ratings by multiplying nP and nQ 
  nR = np.dot(nP, nQ.T)

  # Measure RMSE
  training_rmse = RMSE(df_training_pivot.as_matrix(), nR)
  testing_rmse = RMSE(df_testing_pivot.as_matrix(), nR)

  print("Training RMSE is: {}".format(training_rmse))
  print("Testing RMSE is: {}".format(testing_rmse))  

In [0]:
calculate_recommendation_error(0.02)

Training RMSE is: 0.6062109411039174
Testing RMSE is: 1.2128442479152315


The above shows that our model might have **overfitted** the training data – let's try increasing the `beta` value

In [0]:
calculate_recommendation_error(0.1)

Training RMSE is: 0.5983170937173256
Testing RMSE is: 1.1813741977649694


Difference betweeen $Train_{error}$ and $Test_{error}$ has slightly reduced – which  means overfitting reduced slightly. 

Let's chose a `beta` value from the other direction

In [0]:
calculate_recommendation_error(0.002)

Training RMSE is: 0.5831247155899282
Testing RMSE is: 1.2483107633093686


**Conclusion:** Looks like 0.1 for `beta` performed the best w.r.t. $Test_{error}$ – and to further improve the generalization of our model, we might need get more data
***

# Building a `recommend_for_user()`

## Approach 1 – Matrix Factorization

In [0]:
def recommend_for_user_approach1(userId, n=5):
  """
    Generate movie recommendations for a user.
    Input: userId (from MovieLens data)
    Output: list of tuples (movieId, predictedRating) of top n movies recommended for user (not previously rated by them)
    Algorithm: Matrix Factorization
  """
  df_pivot = df.pivot_table(index='userid', columns='movieid', values='rating', fill_value=0)
  
  # calculate predicted_ratings using Matrix Factorization
  R = df_pivot.as_matrix()
  N = len(R)
  M = len(R[0])
  K = 2

  P = np.random.rand(N,K) # Matrix of user attributes
  Q = np.random.rand(M,K) # Matrix of movie attributes

  nP, nQ = matrix_factorization(R, P, Q, K)

  prediction_matrix = np.dot(nP, nQ.T)  
  
  predictions_df = pd.DataFrame(prediction_matrix)
  
  user_index = userId-1 # userid is 1-indexed
  predicted_ratings_for_user = predictions_df.iloc[user_index, :]
  
  # contruct a df_predicted_ratings
  df_predicted_ratings = pd.DataFrame()
  df_predicted_ratings['movieid'] = pd.Series(np.arange(start=1, stop=len(predicted_ratings_for_user)))
  df_predicted_ratings['prediction'] = predicted_ratings_for_user 
     
  # pull out unrated movies from df_predicted_ratings
  movie_ratings_for_user = df_pivot.iloc[user_index, :]
  index_of_unrated_movies = np.where(movie_ratings_for_user==0)[0]  
  df_unrated_movies = df_predicted_ratings[df_predicted_ratings['movieid'].isin(index_of_unrated_movies)]
  
  # sort and return top 'n'
  df_unrated_movies_sorted = df_unrated_movies.sort_values(['prediction'], ascending=False)  
  recommendations = df_unrated_movies_sorted.iloc[0:n, :]
  
  return [tuple(each) for each in recommendations.values]    

In [0]:
recommend_for_user_approach1(1, 5)

[(4.0, 4.522066104374743),
 (5.0, 4.4062886345883765),
 (6.0, 4.345357269947607),
 (24.0, 4.271277928129292),
 (3.0, 4.245818734699218)]

## Approach 2 –  Similarity Coefficients

In [0]:
import sklearn.metrics as metrics

def recommend_for_user_approach2(userId, n=5):
  """
    Generate movie recommendations for a user.
    Input: userId (from MovieLens data)
    Output: list of tuples (movieId, predictedRating) of top n movies recommended for user (not previously rated by them)
    Algorithm: Similarity Coefficients
  """
  df_pivot = df.pivot_table(index='userid', columns='movieid', values='rating', fill_value=0)
  
  # find top_similar_users
  user_similarity_matrix = metrics.pairwise.cosine_similarity(df_pivot.as_matrix())  
  np.fill_diagonal(user_similarity_matrix, 0)
  user_similarity_df = pd.DataFrame(user_similarity_matrix)
  user_index = userId-1 # userid is 1-indexed
  user_column = user_similarity_df.iloc[:, user_index]
  sorted_user_column = user_column.sort_values(ascending=False)
  top_similar_users = sorted_user_column[0:5]
  
  # find a set of movies which userId has not rated, but top_similar_users have rated highly
  movie_ratings_for_user = df_pivot.iloc[user_index, :]
  unrated_movies_by_user = np.where(movie_ratings_for_user==0)[0]  
  rated_movies_by_similar_users = df[(df['userid']-1).isin(top_similar_users.index.values)]  
  potential_recommendations = rated_movies_by_similar_users[(rated_movies_by_similar_users['movieid']-1).isin(unrated_movies_by_user)]
  potential_recommendations = potential_recommendations.drop(['userid'], axis=1)
    
  # take mean of the ratings which multiple users have rated
  recommendations = pd.DataFrame(columns=['movieid', 'prediction'])
  for each in potential_recommendations['movieid'].unique():    
    mean_rating = potential_recommendations[potential_recommendations['movieid']==each]['rating'].mean()
    recommendations = recommendations.append({'movieid': each, 'prediction': mean_rating}, ignore_index=True)
    
  
  # return top 'n'
  sorted_recommendations = recommendations.sort_values(['prediction'], ascending=False)  
  top_recommendations = sorted_recommendations.iloc[0:n, :]
  return [tuple(each) for each in top_recommendations.values]  


In [0]:
recommend_for_user_approach2(1, 5)

[(3.0, 5.0), (4.0, 5.0), (16.0, 4.5), (10.0, 4.5), (29.0, 4.0)]

## Compare Approach1 v/s Approach2

In [0]:
# use top30 from Approach1, and see where do recommendations from Approach2 stand
recommend_for_user_approach1(1, 30)

[(5.0, 4.373048314532969),
 (10.0, 4.332982126973686),
 (3.0, 4.314746452633742),
 (26.0, 4.238792403502161),
 (21.0, 4.238014811440288),
 (9.0, 4.218558249133384),
 (24.0, 4.1959034041052234),
 (28.0, 4.173853971767062),
 (1.0, 4.171333951947078),
 (32.0, 4.160773357923883),
 (2.0, 4.154852101621238),
 (6.0, 4.153441072000132),
 (20.0, 3.949270766471165),
 (16.0, 3.940695658701465),
 (14.0, 3.922574170859604),
 (22.0, 3.920081591028009),
 (7.0, 3.8437214039704486),
 (15.0, 3.7466451147124724),
 (30.0, 3.6743393462552274),
 (11.0, 3.612787951180856),
 (33.0, 3.561872211176864),
 (19.0, 3.5439412016656164),
 (29.0, 3.4029586307519484),
 (4.0, 3.3245894629722335),
 (23.0, 3.2356607189488256),
 (18.0, 3.012956330947715),
 (31.0, 2.8703998682144762),
 (17.0, 2.6230653688397463)]

**Observation:**

1.   The 1st movie in Approach2 (movieId = 3) is at 3rd position in Approach1 (rating of **5.0 v/s 4.314746452633742**)
2.   The 2nd movie in Approach2 (movieId = 4) is at 25th position in Approach1 (rating of **5.0 v/s 3.3245894629722335**)
3.   The 3rd movie in Approach2 (movieId = 16) is at 14th position in Approach1 (rating of **4.5 v/s 3.940695658701465**)
4.   The 4th movie in Approach2 (movieId = 10) is at 2nd position in Approach1 (rating of **4.5 v/s 4.332982126973686**)
5.   The 5th movie in Approach2 (movieId = 29) is at 24th position in Approach1 (rating of **4.0 v/s 3.4029586307519484**)



**Conclusion:**

4 of the top5 recommendations from **Approach2** seems to have close enough ratings predicted by **Approach1**